# Data leakage

Core stage, lesson C20. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** find a column that carries the answer into training, ask when each value becomes known, fit scaling on the training part only, and read a suspiciously good score as a warning.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def read(name, **kwargs):
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = read("users.csv", parse_dates=["signup_date"])
events = read("events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds
subs = read("subscriptions.csv", parse_dates=["started_on", "ended_on", "updated_at"])

# Known answer from the dataset README: 21,841 people, 177,379 events and 3,840 subscription rows.
assert (len(users), len(events), len(subs)) == (21841, 177379, 3840)

# The platform of a person is the platform of their signup_complete event; its time is the signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])
users["signup_ts"] = users["user_id"].map(signup["event_ts"])

# The table of lesson C19, unit 1: one row per person who signed up from January 1 to June 16 and has a signup event.
last_day = pd.Timestamp("2026-06-16")
first_buy = events[events["event_name"] == "purchase"].groupby("user_id")["event_ts"].min()
people = users[(users["signup_date"] <= last_day) & users["signup_ts"].notna()].copy()
people["days_to_buy"] = (people["user_id"].map(first_buy) - people["signup_ts"]).dt.total_seconds() / 86400
people["bought14"] = (people["days_to_buy"] <= 14).astype(int)                                      # the target: a purchase within 14 days
people["signup_day"] = (people["signup_date"] - pd.Timestamp("2026-01-01")).dt.days                 # days since January 1: known at signup
people["signup_hour"] = people["signup_ts"].dt.hour                                                 # hour of the day (UTC): known at signup
assert len(people) == 20173 and int(people["bought14"].sum()) == 2427

CATS = ["channel", "platform", "region"]          # the honest features of lesson C19


def make_model(extra=()):
    """The recipe of lesson C19, with optional numeric columns that are scaled inside the pipeline."""
    parts = [("cat", OneHotEncoder(handle_unknown="ignore"), CATS)]
    if extra:
        parts.append(("num", StandardScaler(), list(extra)))
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=1000))


def score(model, df):
    """Brier score of a fitted model on the rows of df: the mean squared miss of its chances."""
    return float(brier_score_loss(df["bought14"], model.predict_proba(df)[:, 1]))


print(f"Data loaded: {len(people):,} people, {int(people['bought14'].sum()):,} of them bought within 14 days.")

## Unit 1: A score that is too good

**Predict first.** A teammate adds three columns to the honest recipe of lesson C19: `n_events` (every event of the person in the events table), `plus_uses` (their `plus_feature_used` events) and `has_plan` (1 when the person has a row in the subscriptions table). With seed 20 the baseline scores 0.1059 on the test part. What will the model with the three columns score: about 0.10, about 0.05 or about 0.01? And what would it score on the same people if the app used it at the moment of signup, where `n_events` is 1 and the other two are 0?

In [ ]:
n_events = events.groupby("user_id").size()                                          # every event of the person, whenever it happened
plus_uses = events[events["event_name"] == "plus_feature_used"].groupby("user_id").size()
latest = subs.sort_values("updated_at").groupby("subscription_id").tail(1)           # the current row of each subscription
people["n_events"] = people["user_id"].map(n_events).fillna(0)
people["plus_uses"] = people["user_id"].map(plus_uses).fillna(0)
people["has_plan"] = people["user_id"].isin(latest["user_id"]).astype(int)           # a join to the subscriptions table
LEAKY = ["n_events", "plus_uses", "has_plan"]
train, test = train_test_split(people, test_size=0.25, random_state=20, stratify=people["bought14"])
base = DummyClassifier(strategy="prior").fit(train, train["bought14"])
print(round(float(brier_score_loss(test["bought14"], base.predict_proba(test)[:, 1])), 4))          # the baseline
honest = make_model().fit(train, train["bought14"])
leaky = make_model(LEAKY).fit(train, train["bought14"])
print(round(score(honest, test), 4), round(score(leaky, test), 4))
at_moment = test.assign(n_events=1, plus_uses=0, has_plan=0)                         # the same people as the app sees them at signup
print(round(score(leaky, at_moment), 4), round(float(leaky.predict_proba(at_moment)[:, 1].max()), 4))      # its score, and its largest chance

## Unit 2: Ask when each value becomes known

**Predict first.** Add the three columns to the honest recipe one at a time. Which one alone brings the Brier score closest to 0? Does any of them leave it near the honest 0.1012?

In [ ]:
one_at_a_time = {col: round(score(make_model([col]).fit(train, train["bought14"]), test), 4) for col in LEAKY}
print(one_at_a_time)

## Unit 3: Fit the scaler on the training part only

**Predict first.** A scaler learns a mean and a spread from the rows it is fitted on. Take `signup_day`, the days since January 1. Fitted on all people or on the training part only, how different are the two means? And does the model's Brier score change at 4 decimals?

In [ ]:
num = ["signup_day"]
sc_all = StandardScaler().fit(people[num])                    # learned from all people, the test part included
sc_train = StandardScaler().fit(train[num])                   # learned from the training part only
print(round(float(sc_all.mean_[0]), 2), round(float(sc_train.mean_[0]), 2))


def brier_with_scaler(scaler):
    enc = OneHotEncoder(handle_unknown="ignore").fit(train[CATS])
    design = lambda d: np.hstack([enc.transform(d[CATS]).toarray(), scaler.transform(d[num])])
    lr = LogisticRegression(max_iter=1000).fit(design(train), train["bought14"])
    return float(brier_score_loss(test["bought14"], lr.predict_proba(design(test))[:, 1]))


print(round(brier_with_scaler(sc_all), 4), round(brier_with_scaler(sc_train), 4))
piped = make_model(num).fit(train, train["bought14"])         # the pipeline fits its scaler on the rows it is given to fit
print(round(score(piped, test), 4))

## Unit 4: When the future differs from the past

**Predict first.** Train on the people who signed up before May 1 and test on those from May 1 to June 16. Is the share of buyers the same in both parts? Is the mean of the late people's scaled `signup_day` about the same through a scaler fitted on all people as through one fitted on the early people only? And does the model still beat the baseline?

In [ ]:
cut = pd.Timestamp("2026-05-01")
early, late = people[people["signup_date"] < cut], people[people["signup_date"] >= cut]
print(len(early), len(late), round(float(early["bought14"].mean()), 4), round(float(late["bought14"].mean()), 4))
sc_all = StandardScaler().fit(people[num])
sc_early = StandardScaler().fit(early[num])
print(round(float(sc_all.transform(late[num]).mean()), 2), round(float(sc_early.transform(late[num]).mean()), 2))
model_t = make_model().fit(early, early["bought14"])
base_t = DummyClassifier(strategy="prior").fit(early, early["bought14"])
print(round(score(model_t, late), 4), round(float(brier_score_loss(late["bought14"], base_t.predict_proba(late)[:, 1])), 4))

## Finish it

### Exercise 1: Finish it

**Task:** **How far from the training part does a scaler see the test part?** Take the people who signed up before March 1 as the training part and the rest as the test part (both given). Fit one scaler on the training part only and one on all people. For each, take the mean of the scaled `signup_day` of the test part. Put the list [mean from the training-part scaler, mean from the all-people scaler], each rounded to 3 decimals, in `answer_1`.

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
early1 = people[people["signup_date"] < "2026-03-01"]     # given: the training part, people who signed up before March 1
late1 = people[people["signup_date"] >= "2026-03-01"]     # given: the test part
# fit one scaler on early1 only and one on all people; for each, the mean of the scaled signup_day of late1
answer_1 = None  # replace None with [mean from the early1 scaler, mean from the all-people scaler], each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [5.168, 0.506], traps={(0.506, 5.168): "The scaler of the training part comes first, the scaler of all people second."}, tol=5e-4)

<details>
<summary>Hint</summary>

A scaler is fitted with `StandardScaler().fit(table[["signup_day"]])` and applied with `.transform(...)`. The mean of an array of scaled values is `.mean()`.

</details>

<details>
<summary>Half-solution</summary>

```python
sc_a = StandardScaler().fit(early1[["signup_day"]])
sc_b = StandardScaler().fit(...)
answer_1 = [round(float(sc_a.transform(late1[["signup_day"]]).mean()), 3), ...]
```

</details>

<details>
<summary>Solution</summary>

```python
sc_a = StandardScaler().fit(early1[["signup_day"]])
sc_b = StandardScaler().fit(people[["signup_day"]])
answer_1 = [round(float(sc_a.transform(late1[["signup_day"]]).mean()), 3), round(float(sc_b.transform(late1[["signup_day"]]).mean()), 3)]
```

Seen through the scaler of the training part, the test people sit 5.168 standard deviations above its mean day: they signed up long after anything it learned from. Seen through the scaler that also saw them, they sit 0.506 above: the same people look five times closer. A scaler that has seen the test part hides how far away the test part is.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Which counts leak, and which only look innocent?** Before you start, say in one sentence what the result looks like: four Brier scores, one close to 0.05, two between 0.07 and 0.10 and one close to the honest 0.1012. Four more columns count a person's events in the events table, as `n_events` did in unit 1: `n_checkouts` (`start_checkout`), `n_views` (`view_paywall` and `paywall_viewed`), `n_cancels` (`cancel_subscription`) and `n_sessions` (`session_start`). Build them, add each one alone to the honest recipe (fitted on the training part of the seed-20 split of unit 1, rebuilt with the new columns), and put a dictionary from each column name to its Brier score on the test part, rounded to 4 decimals, in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a dictionary with four columns as keys.

In [ ]:
_ = check("Exercise 2", "answer_2", {"n_checkouts": 0.0524, "n_views": 0.1001, "n_cancels": 0.0783, "n_sessions": 0.1012}, tol=5e-4)

<details>
<summary>Hint</summary>

Count with `events[events["event_name"] == "start_checkout"].groupby("user_id").size()` and map it onto `people["user_id"]` with `.fillna(0)`, as in unit 1. Split again after the columns exist: `train2, test2 = train_test_split(people, test_size=0.25, random_state=20, stratify=people["bought14"])`.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A dictionary of four scores: one far below the honest 0.1012, two in between, and one that has not moved."*

```python
names = {"n_checkouts": ["start_checkout"], "n_views": ["view_paywall", "paywall_viewed"], "n_cancels": ["cancel_subscription"], "n_sessions": ["session_start"]}
for col, kinds in names.items():
    people[col] = people["user_id"].map(events[events["event_name"].isin(kinds)].groupby("user_id").size()).fillna(0)
```

</details>

<details>
<summary>Solution</summary>

```python
names = {"n_checkouts": ["start_checkout"], "n_views": ["view_paywall", "paywall_viewed"], "n_cancels": ["cancel_subscription"], "n_sessions": ["session_start"]}
for col, kinds in names.items():
    people[col] = people["user_id"].map(events[events["event_name"].isin(kinds)].groupby("user_id").size()).fillna(0)
train2, test2 = train_test_split(people, test_size=0.25, random_state=20, stratify=people["bought14"])
answer_2 = {col: round(score(make_model([col]).fit(train2, train2["bought14"]), test2), 4) for col in names}
```

Checkouts (0.0524) and cancels (0.0783) carry the outcome, and views (0.1001) a little of it. Sessions leave the score at 0.1012, the honest value, yet none of the four is known at signup: all four are counts of what happens afterwards. A leak that does not move the score is still a leak in the pipeline: the question is when the value settles, not what it does to the score.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague adds `plan_at_signup` (`plus` or `free`, chosen on the signup screen) as a feature, because it is final at signup, and reports a Brier score of 0.0876 on an 80 to 20 split with seed 7. The score is not honest. Fix the feature list and set `answer_3` to the honest Brier score on the same split, rounded to 4 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
people["plus_signup"] = (people["plan_at_signup"] == "plus").astype(int)     # colleague: "chosen at signup, so it is known at signup"
train7, test7 = train_test_split(people, test_size=0.2, random_state=7, stratify=people["bought14"])
cols7 = ["plus_signup"]                                                      # colleague's extra feature
model7 = make_model(cols7).fit(train7, train7["bought14"])
answer_3 = round(score(model7, test7), 4)

In [ ]:
_ = check("Exercise 3", "answer_3", 0.1015, traps={0.0876: "That is the colleague's score. Take the feature out and score the same split again."}, tol=5e-5)

<details>
<summary>Hint</summary>

Count the people with `plus_signup == 1`, and the share of them with `bought14 == 1`. What does the column say about them?

</details>

<details>
<summary>Half-solution</summary>

```python
cols7 = [...]
model7 = make_model(cols7).fit(train7, train7["bought14"])
```

</details>

<details>
<summary>Solution</summary>

```python
cols7 = []                                                                   # no extra column: the honest recipe
model7 = make_model(cols7).fit(train7, train7["bought14"])
answer_3 = round(score(model7, test7), 4)
```

The honest score is 0.1015. The column is final at signup, but it is final at the instant of the purchase it predicts: the 353 people with `plus_signup == 1` all bought within 14 days, every one. For them the column is the target under another name, so a feature must settle before the moment, not at it.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")